<a href="https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My chosen lane is Lane 2: Refresh / Content Opportunity Scoring. I frame this as a ranking/scoring task because an editor needs an ordered list of pages to review when time is limited. Each content item will receive a provisional review-priority score based on available signals, with clear reasons for its position in the list. The editor will review each recommendation before deciding whether to update, investigate, or monitor the page. I will begin with a transparent rule-based baseline; whether machine learning improves the ranking remains a question to test.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

My intended target is whether an editor judges a page worth reviewing for a possible content update. The starter dataset does not contain this human-reviewed outcome or measured benefits from updates.

For an initial baseline, I will create a provisional review-priority score using search visibility, declining traffic signals, and time since the last update. This score is a rule-defined proxy for review priority, not an observed outcome or proof that refreshing the page will help.

I will not train a model to reproduce this score and claim that it learned which updates succeed. To evaluate usefulness, the top-ranked pages will need independent review using explicit criteria. Predicting future traffic or measuring the effect of updates would require suitable additional outcome data.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

My proposed success metric is Precision@20: the number of independently confirmed review-worthy pages among the top 20 recommendations, divided by 20.

My provisional goal is at least 0.70, meaning 14 of the top 20 pages are confirmed as worth reviewing. This is a proposed goal, not an achieved result.

Before evaluation, I will define a review checklist covering issues such as outdated information, missing useful content, and mismatch with the page’s intended purpose. Reviewers will need access to the actual content and will assess it without seeing its score. They will not simply reuse the scoring rules as the answer.

I will compare the proposed ranking with a simple baseline that orders eligible pages by time since their last update. The starter dataset alone does not provide the content or independent review labels needed to calculate this metric, so evaluation remains pending.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

One row represents one pseudonymized content item. I will initially include items with at least one search impression during the 90-day window.

To illustrate the proxy, I will assign one point for a declining trend and one point for no update in at least 180 days. Scores range from 0 to 2. These equally weighted rules and the 180-day cutoff are provisional choices, not validated measures of update benefit. Missing update ages receive no age point and are flagged for checking.

The score is a baseline for exploration. It is not a training label, an independently reviewed outcome, or a probability of success. Content IDs identify rows and will not be model inputs.

In [5]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/omsawarkar/omsawarkar/"
    "main/data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(url)

# Keep content items with recorded search visibility.
pages = df.loc[df["impressions_90d"] > 0].copy()

# Check that one content ID represents one row.
assert pages["content_id"].notna().all()
assert pages["content_id"].is_unique

# Keep missing update ages visible.
pages["update_age_missing"] = pages["days_since_last_update"].isna()

# Illustrative baseline: one point for each signal.
pages["decline_point"] = (
    pages["trend_direction"] == "down"
).astype(int)

pages["age_point"] = (
    pages["days_since_last_update"] >= 180
).astype(int)

pages["review_priority_proxy"] = (
    pages["decline_point"] + pages["age_point"]
)

print(f"Eligible content items: {len(pages):,}")
print("One row = one unique content item")
print(
    "Missing update ages:",
    pages["update_age_missing"].sum()
)

# Show the real dataframe using pseudonymized IDs.
display(pages[[
    "content_id",
    "impressions_90d",
    "trend_direction",
    "days_since_last_update",
    "update_age_missing",
    "review_priority_proxy"
]].head(10))

print("Number of items at each proxy score:")
display(
    pages["review_priority_proxy"]
    .value_counts()
    .sort_index()
    .rename_axis("proxy_score")
    .to_frame("item_count")
)

Eligible content items: 30,000
One row = one unique content item
Missing update ages: 0


,content_id,impressions_90d,trend_direction,days_since_last_update,update_age_missing,review_priority_proxy
0,content_304f48230142,3803,down,20,False,1
1,content_a1fb4e703a9e,15320,down,25,False,1
2,content_9aa793d4d895,12581,down,20,False,1
3,content_331d6c4de07b,11751,stable,22,False,0
4,content_d99b7a2d90ca,19140,down,14,False,1
5,content_d4084a4bc775,3970,down,20,False,1
6,content_9a34b442b552,20,down,20,False,1
7,content_a63219c6e95a,1724,stable,22,False,0
8,content_5e6c160719bc,32574,down,20,False,1
9,content_c27558df2b0c,1240,down,104,False,1


Number of items at each proxy score:


,item_count
proxy_score,
0,13646
1,16272
2,82


## 5. Why ML beats a fixed rule here

Whether machine learning beats a fixed rule is not yet established. My current score is a transparent baseline using two signals, and it may be sufficient for an initial review queue.

Machine learning could potentially combine page age, search visibility, engagement, and content characteristics in ways that simple thresholds miss. However, more complexity does not automatically produce better recommendations.

To justify ML, I would need independent review labels or suitable observed outcomes. I would compare its Precision@20 against the baseline on held-out items, keeping clients separate between training and evaluation where possible. A model trained only to reproduce my rule-defined proxy would demonstrate rule imitation, not improved review decisions.

For now, this is a scoring and analysis task. I will retain the simple baseline unless reliable evaluation shows that ML adds value.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.